In [6]:
"""
FILE 2: model_training.ipynb
==============================
Loads feature-engineered CSVs and trains:
  Model 1 → Churn Risk Prediction         (customer data)
  Model 2 → Loyalty Detection              (customer data)
  Model 3 → Fraud Detection                (claims data)

Run feature_Engineering.ipynb FIRST.
"""

import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, roc_auc_score, roc_curve,
                             precision_score, recall_score, f1_score)
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

SEED   = 42
COLORS = ['#2E86AB', '#27AE60', '#E74C3C']
plt.style.use('seaborn-v0_8-whitegrid')
print("Libraries loaded.")


Libraries loaded.


In [7]:
# ════════════════════════════════════════════════════════════════
# HELPERS
# ════════════════════════════════════════════════════════════════

def evaluate_model(name, model, Xtr, Xte, ytr, yte, label=""):
    model.fit(Xtr, ytr)
    preds = model.predict(Xte)
    proba = model.predict_proba(Xte)[:, 1]
    acc   = accuracy_score(yte, preds)
    auc   = roc_auc_score(yte, proba)
    prec  = precision_score(yte, preds, zero_division=0)
    rec   = recall_score(yte, preds, zero_division=0)
    f1    = f1_score(yte, preds, zero_division=0)
    cv    = cross_val_score(model, Xtr, ytr, cv=StratifiedKFold(5), scoring='roc_auc').mean()
    print(f"  [{name:<22}]  Acc={acc:.4f}  AUC={auc:.4f}  F1={f1:.4f}  CV-AUC={cv:.4f}")
    print(classification_report(yte, preds, target_names=[f'Not {label}', label], zero_division=0))
    return model, preds, proba, auc


def plot_roc(ax, results, title):
    for name, yte, proba, color in results:
        fpr, tpr, _ = roc_curve(yte, proba)
        auc = roc_auc_score(yte, proba)
        ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} AUC={auc:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set(xlabel='FPR', ylabel='TPR', title=title)
    ax.legend(loc='lower right', fontsize=8)


def plot_importance(ax, model, features, title, color, top_n=12):
    if hasattr(model, 'feature_importances_'):
        imp = pd.Series(model.feature_importances_, index=features)
    else:
        imp = pd.Series(np.abs(model.coef_[0]), index=features)
    imp.nlargest(top_n).sort_values().plot(kind='barh', ax=ax, color=color)
    ax.set(title=title, xlabel='Importance')


def pick_best(candidates):
    """Return candidate tuple with highest AUC."""
    return max(candidates, key=lambda x: x[0])


In [8]:
# ════════════════════════════════════════════════════════════════
# STEP 1: LOAD FEATURE-ENGINEERED DATA
# ════════════════════════════════════════════════════════════════
print("=" * 60)
print("LOADING FEATURE-ENGINEERED DATA")
print("=" * 60)

try:
    cust_eng   = pd.read_csv("customer_engineered.csv")
    claims_eng = pd.read_csv("claims_engineered.csv")
    with open("engineering_metadata.pkl", "rb") as f:
        meta = pickle.load(f)
    print(f"customer_engineered.csv : {cust_eng.shape}")
    print(f"claims_engineered.csv   : {claims_eng.shape}")
    print(f"premium_75th threshold  : {meta['premium_75th']:.2f}")
except FileNotFoundError as e:
    print(f"\nERROR: {e}")
    print("Run feature_Engineering.ipynb first!")
    raise


LOADING FEATURE-ENGINEERED DATA
customer_engineered.csv : (510, 31)
claims_engineered.csv   : (1000, 78)
premium_75th threshold  : 378.68


In [9]:
# ════════════════════════════════════════════════════════════════
# STEP 2: DEFINE FEATURE COLUMNS
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("FEATURE COLUMNS")
print("=" * 60)

# Customer features (for churn + loyalty models)
EXCLUDE_CUST = {'Customer_ID', 'Policy_Type', 'is_high_value',
                'churn_risk', 'is_loyal', 'Date_Joined'}
FEAT_CUST = [c for c in cust_eng.columns
             if c not in EXCLUDE_CUST
             and cust_eng[c].dtype in [np.float64, np.int64, bool, np.uint8]]

# Claims features (for fraud model)
EXCLUDE_FR = {'is_fraud', 'fraud_reported', 'policy_number', 'insured_zip',
              '_c39', 'policy_bind_date', 'incident_date', 'incident_state',
              'incident_city', 'incident_location', 'auto_make', 'auto_model',
              'insured_hobbies'}
FEAT_FR = [c for c in claims_eng.columns
           if c not in EXCLUDE_FR
           and claims_eng[c].dtype in [np.float64, np.int64, np.uint8, bool]]

print(f"Customer features ({len(FEAT_CUST)}): {FEAT_CUST}")
print(f"\nClaims features   ({len(FEAT_FR)}): {FEAT_FR}")



FEATURE COLUMNS
Customer features (26): ['Age', 'Premium', 'Claim_Count', 'Customer_Satisfaction', 'tenure_years', 'claims_per_year', 'age_x_claims', 'premium_x_satisfaction', 'satisfaction_per_tenure', 'Gender_Female', 'Gender_Male', 'Gender_Other', 'Gender_Unknown', 'Region_East', 'Region_North', 'Region_South', 'Region_West', 'age_group_Young', 'age_group_Mid', 'age_group_Senior', 'premium_tier_Low', 'premium_tier_Mid', 'premium_tier_High', 'satisfaction_group_Low', 'satisfaction_group_Mid', 'satisfaction_group_High']

Claims features   (66): ['months_as_customer', 'age', 'policy_deductable', 'policy_annual_premium', 'umbrella_limit', 'capital-gains', 'capital-loss', 'incident_hour_of_the_day', 'number_of_vehicles_involved', 'bodily_injuries', 'witnesses', 'total_claim_amount', 'injury_claim', 'property_claim', 'vehicle_claim', 'auto_year', 'severity_score', 'is_night_incident', 'is_collision', 'vehicle_age', 'claim_per_premium', 'injury_ratio', 'vehicle_ratio', 'property_ratio', '

In [10]:
# ════════════════════════════════════════════════════════════════
# STEP 3: TARGET VARIABLES
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("TARGET VARIABLES")
print("=" * 60)

# Target 1: Churn Risk
# Already computed in feature engineering — column 'churn_risk'
print(f"Churn-risk customers : {cust_eng['churn_risk'].sum()} "
      f"({cust_eng['churn_risk'].mean()*100:.1f}%)")

# Target 2: Loyalty
# Already computed in feature engineering — column 'is_loyal'
print(f"Loyal customers      : {cust_eng['is_loyal'].sum()} "
      f"({cust_eng['is_loyal'].mean()*100:.1f}%)")

# Target 3: Fraud
print(f"Fraud in claims      : {claims_eng['is_fraud'].sum()} "
      f"({claims_eng['is_fraud'].mean()*100:.1f}%)")



TARGET VARIABLES
Churn-risk customers : 307 (60.2%)
Loyal customers      : 120 (23.5%)
Fraud in claims      : 247 (24.7%)


In [11]:
# ════════════════════════════════════════════════════════════════
# MODEL 1: CHURN RISK PREDICTION
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("MODEL 1: CHURN RISK PREDICTION")
print("=" * 60)

X1 = cust_eng[FEAT_CUST].fillna(0)
y1 = cust_eng['churn_risk']

X1_tr, X1_te, y1_tr, y1_te = train_test_split(
    X1, y1, test_size=0.2, random_state=SEED, stratify=y1)

sc1 = StandardScaler()
X1_tr_s = sc1.fit_transform(X1_tr)
X1_te_s = sc1.transform(X1_te)

lr1, lr1_p, lr1_pr, lr1_auc = evaluate_model("Logistic Regression",
    LogisticRegression(max_iter=2000, class_weight='balanced', C=1.0, random_state=SEED),
    X1_tr_s, X1_te_s, y1_tr, y1_te, "Churn")

rf1, rf1_p, rf1_pr, rf1_auc = evaluate_model("Random Forest",
    RandomForestClassifier(n_estimators=300, max_depth=8,
                           class_weight='balanced', min_samples_leaf=3, random_state=SEED),
    X1_tr_s, X1_te_s, y1_tr, y1_te, "Churn")

gb1, gb1_p, gb1_pr, gb1_auc = evaluate_model("Gradient Boosting",
    GradientBoostingClassifier(n_estimators=300, max_depth=4,
                               learning_rate=0.05, subsample=0.8, random_state=SEED),
    X1_tr_s, X1_te_s, y1_tr, y1_te, "Churn")

best1_auc, best1_mdl, best1_lbl, best1_pr, best1_p = pick_best([
    (lr1_auc, lr1, "Logistic Reg",      lr1_pr, lr1_p),
    (rf1_auc, rf1, "Random Forest",     rf1_pr, rf1_p),
    (gb1_auc, gb1, "Gradient Boosting", gb1_pr, gb1_p)])
print(f"\n  ★ Best Model 1 (Churn): {best1_lbl}  AUC={best1_auc:.4f}")



MODEL 1: CHURN RISK PREDICTION
  [Logistic Regression   ]  Acc=0.9216  AUC=0.9844  F1=0.9344  CV-AUC=0.9489
              precision    recall  f1-score   support

   Not Churn       0.90      0.90      0.90        41
       Churn       0.93      0.93      0.93        61

    accuracy                           0.92       102
   macro avg       0.92      0.92      0.92       102
weighted avg       0.92      0.92      0.92       102

  [Random Forest         ]  Acc=1.0000  AUC=1.0000  F1=1.0000  CV-AUC=1.0000
              precision    recall  f1-score   support

   Not Churn       1.00      1.00      1.00        41
       Churn       1.00      1.00      1.00        61

    accuracy                           1.00       102
   macro avg       1.00      1.00      1.00       102
weighted avg       1.00      1.00      1.00       102

  [Gradient Boosting     ]  Acc=1.0000  AUC=1.0000  F1=1.0000  CV-AUC=0.9996
              precision    recall  f1-score   support

   Not Churn       1.00     

In [12]:
# ════════════════════════════════════════════════════════════════
# MODEL 2: LOYALTY DETECTION
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("MODEL 2: LOYALTY DETECTION")
print("=" * 60)

X2 = cust_eng[FEAT_CUST].fillna(0)
y2 = cust_eng['is_loyal']

X2_tr, X2_te, y2_tr, y2_te = train_test_split(
    X2, y2, test_size=0.2, random_state=SEED, stratify=y2)

sc2 = StandardScaler()
X2_tr_s = sc2.fit_transform(X2_tr)
X2_te_s = sc2.transform(X2_te)

lr2, lr2_p, lr2_pr, lr2_auc = evaluate_model("Logistic Regression",
    LogisticRegression(max_iter=2000, class_weight='balanced', C=1.0, random_state=SEED),
    X2_tr_s, X2_te_s, y2_tr, y2_te, "Loyal")

rf2, rf2_p, rf2_pr, rf2_auc = evaluate_model("Random Forest",
    RandomForestClassifier(n_estimators=300, max_depth=8,
                           class_weight='balanced', min_samples_leaf=3, random_state=SEED),
    X2_tr_s, X2_te_s, y2_tr, y2_te, "Loyal")

gb2, gb2_p, gb2_pr, gb2_auc = evaluate_model("Gradient Boosting",
    GradientBoostingClassifier(n_estimators=300, max_depth=4,
                               learning_rate=0.05, subsample=0.8, random_state=SEED),
    X2_tr_s, X2_te_s, y2_tr, y2_te, "Loyal")

best2_auc, best2_mdl, best2_lbl, best2_pr, best2_p = pick_best([
    (lr2_auc, lr2, "Logistic Reg",      lr2_pr, lr2_p),
    (rf2_auc, rf2, "Random Forest",     rf2_pr, rf2_p),
    (gb2_auc, gb2, "Gradient Boosting", gb2_pr, gb2_p)])
print(f"\n  ★ Best Model 2 (Loyalty): {best2_lbl}  AUC={best2_auc:.4f}")



MODEL 2: LOYALTY DETECTION
  [Logistic Regression   ]  Acc=0.8824  AUC=0.9728  F1=0.7778  CV-AUC=0.9325
              precision    recall  f1-score   support

   Not Loyal       0.96      0.88      0.92        78
       Loyal       0.70      0.88      0.78        24

    accuracy                           0.88       102
   macro avg       0.83      0.88      0.85       102
weighted avg       0.90      0.88      0.89       102

  [Random Forest         ]  Acc=1.0000  AUC=1.0000  F1=1.0000  CV-AUC=0.9993
              precision    recall  f1-score   support

   Not Loyal       1.00      1.00      1.00        78
       Loyal       1.00      1.00      1.00        24

    accuracy                           1.00       102
   macro avg       1.00      1.00      1.00       102
weighted avg       1.00      1.00      1.00       102

  [Gradient Boosting     ]  Acc=0.9902  AUC=1.0000  F1=0.9796  CV-AUC=0.9997
              precision    recall  f1-score   support

   Not Loyal       1.00      0.9

In [13]:
# ════════════════════════════════════════════════════════════════
# MODEL 3: FRAUD DETECTION
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("MODEL 3: FRAUD DETECTION")
print("=" * 60)

X3 = claims_eng[FEAT_FR].fillna(claims_eng[FEAT_FR].median())
y3 = claims_eng['is_fraud']

print(f"Fraud features : {len(FEAT_FR)}")
print(f"Class balance  : {y3.value_counts().to_dict()}")

X3_tr, X3_te, y3_tr, y3_te = train_test_split(
    X3, y3, test_size=0.2, random_state=SEED, stratify=y3)

sc3 = StandardScaler()
X3_tr_s = sc3.fit_transform(X3_tr)
X3_te_s = sc3.transform(X3_te)

lr3, lr3_p, lr3_pr, lr3_auc = evaluate_model("Logistic Regression",
    LogisticRegression(max_iter=2000, class_weight='balanced', C=0.5, random_state=SEED),
    X3_tr_s, X3_te_s, y3_tr, y3_te, "Fraud")

rf3, rf3_p, rf3_pr, rf3_auc = evaluate_model("Random Forest",
    RandomForestClassifier(n_estimators=300, max_depth=10,
                           class_weight='balanced', min_samples_leaf=2, random_state=SEED),
    X3_tr_s, X3_te_s, y3_tr, y3_te, "Fraud")

gb3, gb3_p, gb3_pr, gb3_auc = evaluate_model("Gradient Boosting",
    GradientBoostingClassifier(n_estimators=300, max_depth=5,
                               learning_rate=0.05, subsample=0.8, random_state=SEED),
    X3_tr_s, X3_te_s, y3_tr, y3_te, "Fraud")

best3_auc, best3_mdl, best3_lbl, best3_pr, best3_p = pick_best([
    (lr3_auc, lr3, "Logistic Reg",      lr3_pr, lr3_p),
    (rf3_auc, rf3, "Random Forest",     rf3_pr, rf3_p),
    (gb3_auc, gb3, "Gradient Boosting", gb3_pr, gb3_p)])
print(f"\n  ★ Best Model 3 (Fraud): {best3_lbl}  AUC={best3_auc:.4f}")



MODEL 3: FRAUD DETECTION
Fraud features : 66
Class balance  : {0: 753, 1: 247}
  [Logistic Regression   ]  Acc=0.8350  AUC=0.8102  F1=0.6916  CV-AUC=0.7352
              precision    recall  f1-score   support

   Not Fraud       0.92      0.86      0.89       151
       Fraud       0.64      0.76      0.69        49

    accuracy                           0.83       200
   macro avg       0.78      0.81      0.79       200
weighted avg       0.85      0.83      0.84       200

  [Random Forest         ]  Acc=0.8150  AUC=0.7769  F1=0.6186  CV-AUC=0.7492
              precision    recall  f1-score   support

   Not Fraud       0.88      0.88      0.88       151
       Fraud       0.62      0.61      0.62        49

    accuracy                           0.81       200
   macro avg       0.75      0.75      0.75       200
weighted avg       0.81      0.81      0.81       200

  [Gradient Boosting     ]  Acc=0.7600  AUC=0.7594  F1=0.4000  CV-AUC=0.7331
              precision    recall  

In [14]:
# ════════════════════════════════════════════════════════════════
# DASHBOARD PLOTS
# ════════════════════════════════════════════════════════════════
print("\nGenerating dashboard...")
fig = plt.figure(figsize=(20, 18))
fig.suptitle('Insurance Analytics — Model Dashboard',
             fontsize=18, fontweight='bold', y=0.98)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

plot_roc(fig.add_subplot(gs[0, 0]),
         [("Log Reg", y1_te, lr1_pr, COLORS[0]),
          ("Rand For", y1_te, rf1_pr, COLORS[1]),
          ("Grad Bst", y1_te, gb1_pr, COLORS[2])],
         "ROC — Churn Risk")
plot_roc(fig.add_subplot(gs[1, 0]),
         [("Log Reg", y2_te, lr2_pr, COLORS[0]),
          ("Rand For", y2_te, rf2_pr, COLORS[1]),
          ("Grad Bst", y2_te, gb2_pr, COLORS[2])],
         "ROC — Loyalty")
plot_roc(fig.add_subplot(gs[2, 0]),
         [("Log Reg", y3_te, lr3_pr, COLORS[0]),
          ("Rand For", y3_te, rf3_pr, COLORS[1]),
          ("Grad Bst", y3_te, gb3_pr, COLORS[2])],
         "ROC — Fraud Detection")

plot_importance(fig.add_subplot(gs[0, 1]), best1_mdl, FEAT_CUST,
                f"Top Features — Churn\n(Best: {best1_lbl})", COLORS[0])
plot_importance(fig.add_subplot(gs[1, 1]), best2_mdl, FEAT_CUST,
                f"Top Features — Loyalty\n(Best: {best2_lbl})", COLORS[1])
plot_importance(fig.add_subplot(gs[2, 1]), best3_mdl, FEAT_FR,
                f"Top Features — Fraud\n(Best: {best3_lbl})", COLORS[2])

for ax, yte, pred, title, cmap in [
    (fig.add_subplot(gs[0, 2]), y1_te, best1_p,
     f"Confusion — Churn\nAUC={best1_auc:.3f}", "Blues"),
    (fig.add_subplot(gs[1, 2]), y2_te, best2_p,
     f"Confusion — Loyalty\nAUC={best2_auc:.3f}", "Greens"),
    (fig.add_subplot(gs[2, 2]), y3_te, best3_p,
     f"Confusion — Fraud\nAUC={best3_auc:.3f}", "Reds"),
]:
    cm = confusion_matrix(yte, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                cbar=False, annot_kws={'size': 13})
    ax.set(title=title, xlabel='Predicted', ylabel='Actual')

plt.savefig('model_dashboard.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: model_dashboard.png")



Generating dashboard...
  Saved: model_dashboard.png


In [15]:
# ════════════════════════════════════════════════════════════════
# SUMMARY TABLE
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)

summary = pd.DataFrame({
    'Model'         : ['Churn Risk', 'Loyalty Detection', 'Fraud Detection'],
    'Best Algorithm': [best1_lbl, best2_lbl, best3_lbl],
    'Test AUC'      : [round(best1_auc, 4), round(best2_auc, 4), round(best3_auc, 4)],
    'Accuracy'      : [round(accuracy_score(y1_te, best1_p), 4),
                       round(accuracy_score(y2_te, best2_p), 4),
                       round(accuracy_score(y3_te, best3_p), 4)],
    'F1 Score'      : [round(f1_score(y1_te, best1_p, zero_division=0), 4),
                       round(f1_score(y2_te, best2_p, zero_division=0), 4),
                       round(f1_score(y3_te, best3_p, zero_division=0), 4)],
})
print(summary.to_string(index=False))
summary.to_csv("model_summary.csv", index=False)
print("\nSaved: model_summary.csv")



FINAL SUMMARY
            Model Best Algorithm  Test AUC  Accuracy  F1 Score
       Churn Risk  Random Forest    1.0000     1.000    1.0000
Loyalty Detection  Random Forest    1.0000     1.000    1.0000
  Fraud Detection   Logistic Reg    0.8102     0.835    0.6916

Saved: model_summary.csv


In [16]:
# ════════════════════════════════════════════════════════════════
# SAVE TRAINED MODELS TO PKL
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("SAVING TRAINED MODELS")
print("=" * 60)

bundle = {
    # Models
    'model1_churn'   : best1_mdl,    # Churn Risk
    'model2_loyalty' : best2_mdl,    # Loyalty Detection
    'model3_fraud'   : best3_mdl,    # Fraud Detection

    # Labels
    'model1_label'   : best1_lbl,
    'model2_label'   : best2_lbl,
    'model3_label'   : best3_lbl,

    # Scalers (MUST match the model they were fit with)
    'scaler1'        : sc1,
    'scaler2'        : sc2,
    'scaler3'        : sc3,

    # Exact feature column lists
    'feat_cust'      : FEAT_CUST,
    'feat_fr'        : FEAT_FR,

    # Metadata
    'meta'           : meta,
}

PKL_PATH = 'trained_models.pkl'
with open(PKL_PATH, 'wb') as f:
    pickle.dump(bundle, f)

# Sanity reload
with open(PKL_PATH, 'rb') as f:
    _check = pickle.load(f)
assert all(k in _check for k in ('model1_churn', 'model2_loyalty', 'model3_fraud'))

print(f"  Saved : {PKL_PATH}")
print(f"  Churn   — {best1_lbl}   AUC={best1_auc:.4f}")
print(f"  Loyalty — {best2_lbl}   AUC={best2_auc:.4f}")
print(f"  Fraud   — {best3_lbl}   AUC={best3_auc:.4f}")
print(f"  feat_cust : {len(FEAT_CUST)} columns")
print(f"  feat_fr   : {len(FEAT_FR)} columns")
print("\n  ✅  Bundle verified — all 3 models saved successfully.")



SAVING TRAINED MODELS
  Saved : trained_models.pkl
  Churn   — Random Forest   AUC=1.0000
  Loyalty — Random Forest   AUC=1.0000
  Fraud   — Logistic Reg   AUC=0.8102
  feat_cust : 26 columns
  feat_fr   : 66 columns

  ✅  Bundle verified — all 3 models saved successfully.


In [17]:
# ════════════════════════════════════════════════════════════════
# PREDICTION HELPER FUNCTIONS  (reusable anywhere)
# ════════════════════════════════════════════════════════════════

def load_models(path='trained_models.pkl'):
    with open(path, 'rb') as f:
        return pickle.load(f)


def predict_churn(customer_row: dict, bundle: dict) -> dict:
    """customer_row: dict of engineered feature values."""
    inp  = pd.DataFrame([customer_row]).reindex(columns=bundle['feat_cust'], fill_value=0)
    prob = bundle['model1_churn'].predict_proba(bundle['scaler1'].transform(inp))[0][1]
    risk = 'High' if prob >= 0.6 else 'Medium' if prob >= 0.4 else 'Low'
    return {'churn_risk': risk, 'probability': round(float(prob), 4)}


def predict_loyalty(customer_row: dict, bundle: dict) -> dict:
    inp  = pd.DataFrame([customer_row]).reindex(columns=bundle['feat_cust'], fill_value=0)
    prob = bundle['model2_loyalty'].predict_proba(bundle['scaler2'].transform(inp))[0][1]
    tier = 'Champion' if prob >= 0.7 else 'Engaged' if prob >= 0.4 else 'At Risk'
    return {'loyalty_tier': tier, 'probability': round(float(prob), 4)}


def predict_fraud(claim_row: dict, bundle: dict) -> dict:
    inp   = pd.DataFrame([claim_row]).reindex(columns=bundle['feat_fr'], fill_value=0)
    prob  = bundle['model3_fraud'].predict_proba(bundle['scaler3'].transform(inp))[0][1]
    alert = 'HIGH' if prob >= 0.7 else 'MEDIUM' if prob >= 0.4 else 'LOW'
    return {'fraud_likely': bool(prob >= 0.5), 'probability': round(float(prob), 4),
            'alert_level': alert}


print("Helper functions defined: predict_churn, predict_loyalty, predict_fraud")
print("Load models with: bundle = load_models('trained_models.pkl')")


Helper functions defined: predict_churn, predict_loyalty, predict_fraud
Load models with: bundle = load_models('trained_models.pkl')
